# End-to-end demo — video link → clips → base vs LoRA captions

Give it ONE video URL. It runs the full clip-extraction pipeline on that video
(octopus detection with `clip_mlp_hardneg_v2.pt` + absolute motion gate + 20s
non-overlapping windows), saves the found clips to JSON, then captions each clip
with **two models** and stores both:
- `caption_base`  — plain `Qwen2.5-VL-3B-Instruct`
- `caption_lora`  — the same base + your fine-tuned LoRA adapter (the caption student)

> Runtime → A100/L4 GPU. Needs on Drive: `clip_mlp_hardneg_v2.pt` and your adapter zip
> (`caption_student_qwen25vl3b_lora.zip` from the training notebook).

## 1. Install

In [ ]:
!pip -q install -U "transformers>=4.49" peft accelerate qwen-vl-utils openai-clip
!apt-get -qq install -y ffmpeg >/dev/null
print("Installed. If a torch/CUDA error appears: Runtime -> Restart, then run from CONFIG.")

In [ ]:
import torch; print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

## 2. Config

In [ ]:
from pathlib import Path

# --- the video to process ---
VIDEO_URL   = "https://repo.octopus-intelligence.org/public/O-vulgaris-Nity-2026-2-20--/Right%20Front/Local/2026-02-20/172422--vv-1.mp4"
OCTOPUS_USER = "octopus"          # server creds (embedded into the URL); "" if the URL is public
OCTOPUS_PASS = ""                 # <-- fill in, or leave blank for a public URL

# --- extraction gates (match extract_octopus_clips.py) ---
SAMPLE_FPS       = 1.0
CLIP_LEN         = 20
VIS_THRESH       = 0.6      # per-frame p_visible to count as octopus-visible
MIN_VISIBLE_FRAC = 0.5      # >this fraction of window frames must be visible
MOTION_THRESH    = 0.008    # mean absolute changed-pixel fraction per window
MOTION_PIX       = 25

# --- captioning ---
BASE_MODEL   = "Qwen/Qwen2.5-VL-3B-Instruct"
N_FRAMES     = 4
MAX_PIXELS   = 360*420
GEN_CACHE    = False        # DynamicCache concat can be buggy for this model; False is safe
PROMPT = ("These frames are sampled in order from one short aquarium clip of Nity, an octopus. "
          "Write ONE sentence describing what the octopus does across the clip, "
          "or 'octopus not present' if no octopus is visible.")

# --- paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/GSOC-Catrobat")
CLIP_CKPT   = Path("clip_mlp_hardneg_v2.pt")
ADAPTER_DIR = Path("caption_student_qwen25vl3b_lora")
CLIPS_DIR   = Path("extracted_clips"); CLIPS_DIR.mkdir(exist_ok=True)
OUT_JSON    = Path("demo_clips_captions.json")

## 3. Get model files from Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import shutil, zipfile
shutil.copy(DRIVE_ROOT / CLIP_CKPT.name, CLIP_CKPT)
if not ADAPTER_DIR.exists():
    with zipfile.ZipFile(DRIVE_ROOT / (ADAPTER_DIR.name + ".zip")) as z:
        z.extractall(ADAPTER_DIR)
print("ckpt + adapter ready:", CLIP_CKPT.exists(), ADAPTER_DIR.exists())

## 4. Extraction — octopus detector + motion + 20s windows

In [ ]:
import subprocess, numpy as np
from PIL import Image
import torch, torch.nn as nn
try:
    import pkg_resources, packaging, packaging.version, packaging.specifiers, packaging.requirements
    pkg_resources.packaging = packaging
except Exception: pass
import clip as clip_lib

dev = "cuda"
def auth(u): return u.replace("https://", f"https://{OCTOPUS_USER}:{OCTOPUS_PASS}@") if OCTOPUS_PASS else u

def letterbox(img, size=224, fill=(128,128,128)):
    w,h=img.size; s=size/max(w,h); nw,nh=max(1,round(w*s)),max(1,round(h*s))
    img=img.resize((nw,nh),Image.BICUBIC); cv=Image.new("RGB",(size,size),fill)
    cv.paste(img,((size-nw)//2,(size-nh)//2)); return cv

# octopus detector (CLIP ViT-B/32 + MLP probe)
_ck=torch.load(CLIP_CKPT, map_location=dev)
clip_model,_pre=clip_lib.load(_ck["clip_model"], device=dev); clip_model.eval()
def _clf(ck):
    feat=ck["feat_dim"]; hid=[int(x) for x in ck["arch"].replace("mlp_","").split("_")]; dims=[feat]+hid+[2]; L=[]
    for i in range(len(dims)-1):
        L.append(nn.Linear(dims[i],dims[i+1]))
        if i<len(dims)-2: L+=[nn.ReLU(),nn.Dropout(0.3)]
    return nn.Sequential(*L)
det=_clf(_ck).to(dev); det.load_state_dict(_ck["state_dict"]); det.eval()
VIS_IDX=_ck.get("label_map",{}).get("visible",1)

def octopus_pv(url):
    """per-second p_visible via one 1fps letterboxed ffmpeg pass"""
    cmd=["ffmpeg","-loglevel","error","-i",auth(url),
         "-vf",f"fps={SAMPLE_FPS},scale=224:224:force_original_aspect_ratio=decrease,pad=224:224:-1:-1:color=gray",
         "-f","image2pipe","-vcodec","rawvideo","-pix_fmt","rgb24","-"]
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.PIPE); fs=224*224*3; pv=[]; buf=[]
    def flush():
        if not buf: return
        with torch.no_grad():
            b=torch.stack([_pre(im) for im in buf]).to(dev)
            f=clip_model.encode_image(b).float(); f=f/f.norm(dim=-1,keepdim=True)
            pv.extend(torch.softmax(det(f),1)[:,VIS_IDX].cpu().tolist())
        buf.clear()
    while True:
        raw=p.stdout.read(fs)
        if len(raw)<fs: break
        buf.append(letterbox(Image.fromarray(np.frombuffer(raw,np.uint8).reshape(224,224,3))))
        if len(buf)>=64: flush()
    flush(); p.stdout.close(); p.wait(); return np.array(pv,np.float32)

def motion_frac(url):
    """per-second absolute changed-pixel fraction (scan_motion_area), timestamp masked"""
    cmd=["ffmpeg","-loglevel","error","-i",auth(url),"-vf",f"fps={SAMPLE_FPS},scale=224:224,format=gray",
         "-f","image2pipe","-vcodec","rawvideo","-pix_fmt","gray","-"]
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.PIPE); fs=224*224; prev=None; out=[]
    while True:
        raw=p.stdout.read(fs)
        if len(raw)<fs: break
        f=np.frombuffer(raw,np.uint8).reshape(224,224).astype(np.float32)
        if prev is not None:
            d=np.abs(f-prev); d[int(224*0.88):, int(224*0.60):]=0.0
            out.append(float((d>MOTION_PIX).mean()))
        prev=f
    p.stdout.close(); p.wait(); return np.array(out,np.float32)

def find_windows(pv, mot):
    L=int(CLIP_LEN*SAMPLE_FPS); N=len(pv); m=np.zeros(N,np.float32)
    m[1:1+len(mot)]=mot[:max(0,N-1)]            # align motion (starts at 2nd frame) to pv grid
    out=[]; s=0
    while s+L<=N:
        vf=float((pv[s:s+L]>=VIS_THRESH).mean()); mm=float(m[s:s+L].mean())
        if vf>MIN_VISIBLE_FRAC and mm>=MOTION_THRESH:
            out.append({"start_sec":s,"end_sec":s+L,"visible_frac":round(vf,3),"mean_motion":round(mm,5)}); s+=L
        else: s+=1
    return out

def hhmmss(x): return f"{x//60:02d}:{x%60:02d}"
def extract(url,s,e,path):
    subprocess.run(["ffmpeg","-loglevel","error","-y","-ss",str(s),"-to",str(e),"-i",auth(url),
                    "-c","copy",str(path)],capture_output=True)
    return path.exists() and path.stat().st_size>10000
print("extraction helpers ready.")

In [ ]:
import json
print("scanning video (octopus + motion passes)...", flush=True)
pv=octopus_pv(VIDEO_URL); mot=motion_frac(VIDEO_URL)
print(f"  {len(pv)} sampled seconds | visible>= {VIS_THRESH}: {(pv>=VIS_THRESH).sum()} | motion mean {mot.mean():.4f}")
wins=find_windows(pv,mot)
print(f"  -> {len(wins)} clips pass both gates")
clips=[]
for w in wins:
    path=CLIPS_DIR/f"clip_{w['start_sec']:04d}-{w['end_sec']:04d}.mp4"
    if extract(VIDEO_URL,w["start_sec"],w["end_sec"],path):
        clips.append({**w,"video_url":VIDEO_URL,"video_timeline":f"{hhmmss(w['start_sec'])}-{hhmmss(w['end_sec'])}",
                      "clip_path":str(path)})
json.dump({"video_url":VIDEO_URL,"count":len(clips),"clips":clips}, open(OUT_JSON,"w"), indent=2)
print(f"extracted {len(clips)} clips -> {OUT_JSON}")

## 5. Load Qwen2.5-VL-3B (base) + LoRA adapter

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

processor=AutoProcessor.from_pretrained(BASE_MODEL, max_pixels=MAX_PIXELS)
base=Qwen2_5_VLForConditionalGeneration.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")
base.config.use_cache=GEN_CACHE; base.eval()
print("base loaded.")

## 6. Caption helper + frame sampling

In [ ]:
import subprocess
from pathlib import Path as P

def frames_for(clip_path):
    import tempfile, numpy as np
    from PIL import Image
    t=tempfile.mkdtemp()
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",clip_path,"-vf","fps=1,scale='min(640,iw)':-2","-q:v","3",f"{t}/f_%03d.jpg"],capture_output=True)
    fs=sorted(P(t).glob("f_*.jpg"))
    if not fs: return []
    idx=np.linspace(0,len(fs)-1,min(N_FRAMES,len(fs))).round().astype(int)
    return [str(fs[i]) for i in idx]

def caption(model, frame_paths):
    content=[{"type":"image","image":f,"max_pixels":MAX_PIXELS} for f in frame_paths]+[{"type":"text","text":PROMPT}]
    msgs=[{"role":"user","content":content}]
    text=processor.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    im,_=process_vision_info(msgs)
    inp=processor(text=[text],images=[im],return_tensors="pt").to(model.device)
    inp.pop("mm_token_type_ids",None)
    with torch.no_grad():
        out=model.generate(**inp,max_new_tokens=80,do_sample=False,use_cache=GEN_CACHE)
    return processor.batch_decode(out[:,inp["input_ids"].shape[1]:],skip_special_tokens=True)[0].strip()
print("caption helpers ready.")

## 7. Caption every clip with BASE, then with LoRA

In [ ]:
import json
data=json.load(open(OUT_JSON))
# precompute frames once per clip
for c in data["clips"]:
    c["_frames"]=frames_for(c["clip_path"])

print("captioning with BASE model...", flush=True)
for i,c in enumerate(data["clips"],1):
    c["caption_base"]=caption(base,c["_frames"]) if c["_frames"] else "(no frames)"
    print(f"  [{i}/{len(data['clips'])}] base: {c['caption_base'][:80]}")

print("\nattaching LoRA adapter...", flush=True)
from peft import PeftModel
lora=PeftModel.from_pretrained(base, str(ADAPTER_DIR)); lora.config.use_cache=GEN_CACHE; lora.eval()

print("captioning with LoRA student...", flush=True)
for i,c in enumerate(data["clips"],1):
    c["caption_lora"]=caption(lora,c["_frames"]) if c["_frames"] else "(no frames)"
    print(f"  [{i}/{len(data['clips'])}] lora: {c['caption_lora'][:80]}")

for c in data["clips"]: c.pop("_frames",None)
json.dump(data, open(OUT_JSON,"w"), indent=2)
print(f"\nsaved base+lora captions -> {OUT_JSON}")

## 8. Compare + save to Drive

In [ ]:
import json
d=json.load(open(OUT_JSON))
for c in d["clips"]:
    print(f"[{c['video_timeline']}] vis={c['visible_frac']} motion={c['mean_motion']}")
    print(f"  BASE: {c.get('caption_base')}")
    print(f"  LORA: {c.get('caption_lora')}")
    print("-"*70)
import shutil
shutil.copy(OUT_JSON, DRIVE_ROOT/OUT_JSON.name)
print("saved ->", DRIVE_ROOT/OUT_JSON.name)